<a href="https://colab.research.google.com/github/kumar-git/ProgrammingAssignment2/blob/master/AML_CoPilot_Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langgraph faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.1 MB/s eta 0:00:00


In [3]:
# ===========================================================
# LangGraph‑Orchestrated Agentic AML Investigation Copilot
# ===========================================================

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
import faiss
import torch
import hf_xet
import accelerate

# LangGraph imports
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Hugging Face imports — causal LLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [4]:
# -----------------------------------------------------------
# 0. Define Shared State
# -----------------------------------------------------------

class AMLState(TypedDict):
    customer_id: str
    feature_row: list[float]
    risk_score: float
    drivers: dict
    patterns: list[str]
    context: str
    typologies: list[str]
    guidance: str
    recommendation: str

In [5]:
# -----------------------------------------------------------
# 1. Synthetic AML Data
# -----------------------------------------------------------

def generate_data():
    np.random.seed(42)
    customers = pd.DataFrame({
        'customer_id': [f"CUST{i}" for i in range(500)],
        'risk_rating': np.random.choice(['Low','Medium','High'], 500),
        'country': np.random.choice(['US','IN','GB','SG'], 500),
        'kyc_score': np.random.rand(500)
    })

    transactions = pd.DataFrame({
        'customer_id': np.random.choice(customers['customer_id'], 5000),
        'amount': np.random.uniform(10,20000,5000),
        'cash_flag': np.random.choice([0,1],5000,p=[0.8,0.2]),
    })

    return customers, transactions

customers, transactions = generate_data()

features = transactions.groupby('customer_id').agg(
    txn_count=('amount','count'),
    txn_sum=('amount','sum'),
    cash_ratio=('cash_flag','mean')
).reset_index()

In [6]:
# -----------------------------------------------------------
# 2. ML Models (Risk Scoring + Pattern Detection)
# -----------------------------------------------------------

features['label'] = np.random.choice([0,1], len(features), p=[0.7,0.3])
X = features[['txn_count','txn_sum','cash_ratio']]
y = features['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

risk_model = XGBClassifier(eval_metric='logloss')
risk_model.fit(X_scaled, y)

pattern_model = IsolationForest(contamination=0.05)
pattern_model.fit(X_scaled)

IsolationForest(contamination=0.05)

In [7]:
# -----------------------------------------------------------
# 3. RAG Embeddings (Customer Context) via FAISS
# -----------------------------------------------------------

embed_model = SentenceTransformer('all-mpnet-base-v2')
docs = [
    f"KYC score {r.kyc_score}, country {r.country}, risk {r.risk_rating}"
    for _,r in customers.iterrows()
]
embs = embed_model.encode(docs, convert_to_numpy=True).astype('float32')
faiss_index = faiss.IndexFlatL2(embs.shape[1])
faiss_index.add(embs)
cid_to_idx = {cid:i for i,cid in enumerate(customers['customer_id'])}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# -----------------------------------------------------------
# Login to HuggingFace using generated Token on HuggingFace website
# -----------------------------------------------------------

from huggingface_hub import notebook_login
notebook_login()

In [9]:
pip install -U bitsandbytes>=0.46.1

In [10]:
# -----------------------------------------------------------
# 4. LLM (Hugging Face)
# -----------------------------------------------------------

# Correct Configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

#model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
model_id = "tiiuae/falcon-7b-instruct" ## Very heavy - ran for 12 hrs on local system, then has been stopped. Ran on GPU in Google Colab in 12 minutes.
#model_id = "HuggingFaceTB/SmolLM2-135M-Instruct" ## Working fine, fast
#model_id = "meta-llama/Llama-3.2-1B-Instruct" # access through HF login

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

def run_llm(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_length=300)
    return tokenizer.decode(out[0], skip_special_tokens=True)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [11]:
# -----------------------------------------------------------
# 5. Graph Node Functions
# -----------------------------------------------------------

def risk_scoring_node(state: AMLState) -> AMLState:
    feat = state['feature_row']
    score = float(risk_model.predict_proba(scaler.transform([feat]))[0][1])
    drivers = dict(zip(['txn_count','txn_sum','cash_ratio'], feat))
    state['risk_score'] = score
    state['drivers'] = drivers
    return state

def pattern_analysis_node(state: AMLState) -> AMLState:
    feat = state['feature_row']
    pred = pattern_model.predict(scaler.transform([feat]))[0]
    state['patterns'] = ['Anomaly'] if pred == -1 else []
    return state

def customer_context_node(state: AMLState) -> AMLState:
    cid = state['customer_id']
    idx = cid_to_idx[cid]
    emb = embs[idx].reshape(1,-1)
    _, nbrs = faiss_index.search(emb,3)
    context = " | ".join([docs[n] for n in nbrs[0]])
    state['context'] = context
    return state

def typology_mapping_node(state: AMLState) -> AMLState:
    state['typologies'] = ['Structuring'] if state['patterns'] else []
    return state

def regulatory_reasoning_node(state: AMLState) -> AMLState:
    prompt = (
        f"Customer context: {state['context']}\n"
        f"Patterns: {state['patterns']}\n"
        #"Provide fact‑based AML guidance."
        "Generate a SAR narrative justifying escalation"
    )
    state['guidance'] = run_llm(prompt)
    return state

def decision_node(state: AMLState) -> AMLState:
    state['recommendation'] = (
        "Escalate for SAR"
        if state['risk_score']>0.7 or state['typologies']
        else "Clear"
    )
    return state

In [12]:
# -----------------------------------------------------------
# 6. Build LangGraph Workflow
# -----------------------------------------------------------

graph = StateGraph(AMLState)

graph.add_node("risk_scoring", risk_scoring_node)
graph.add_node("pattern_analysis", pattern_analysis_node)
graph.add_node("customer_context", customer_context_node)
graph.add_node("typology_mapping", typology_mapping_node)
graph.add_node("regulatory_reasoning", regulatory_reasoning_node)
graph.add_node("decision", decision_node)

graph.add_edge(START, "risk_scoring")
graph.add_edge("risk_scoring", "pattern_analysis")
graph.add_edge("pattern_analysis", "customer_context")
graph.add_edge("customer_context", "typology_mapping")
graph.add_edge("typology_mapping", "regulatory_reasoning")
graph.add_edge("regulatory_reasoning", "decision")
graph.add_edge("decision", END)

compiled = graph.compile()

In [13]:
# -----------------------------------------------------------
# 7. Batch Processing
# -----------------------------------------------------------

from typing import List

def investigate_batch(customer_ids: List[str]):
    results = []
    for cid in customer_ids:
        feat_row = features.loc[features['customer_id']==cid,
                                ['txn_count','txn_sum','cash_ratio']].values[0].tolist()
        initial_state: AMLState = {
            'customer_id': cid,
            'feature_row': feat_row,
            'risk_score': 0,
            'drivers': {},
            'patterns': [],
            'context': "",
            'typologies': [],
            'guidance': "",
            'recommendation': ""
        }
        outcome = compiled.invoke(initial_state)
        results.append(outcome)
    return results

In [14]:
# -----------------------------------------------------------
# 8. Demo Run
# -----------------------------------------------------------

if __name__ == "__main__":
    top_ids = features.sort_values(by='txn_sum', ascending=False).head(3)['customer_id'].tolist()
    investigations = investigate_batch(top_ids)
    for res in investigations:
        print("\nCustomer:", res['customer_id'])
        print("Rec:", res['recommendation'])
        print("Risk:", round(res['risk_score'],3))
        print("LLM Guidance:", res['guidance'][:200])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have


Customer: CUST386
Rec: Escalate for SAR
Risk: 0.541
LLM Guidance: Customer context: KYC score 0.41279355060675227, country SG, risk High | KYC score 0.6332407749109605, country SG, risk High | KYC score 0.7184572708365536, country SG, risk High
Patterns: ['Anomaly']

Customer: CUST312
Rec: Escalate for SAR
Risk: 0.212
LLM Guidance: Customer context: KYC score 0.03422811082052912, country SG, risk High | KYC score 0.41279355060675227, country SG, risk High | KYC score 0.5639650920188364, country SG, risk High
Patterns: ['Anomaly'

Customer: CUST424
Rec: Escalate for SAR
Risk: 0.203
LLM Guidance: Customer context: KYC score 0.11528202124728437, country GB, risk Low | KYC score 0.9103391020575343, country GB, risk Low | KYC score 0.13911619414306187, country GB, risk Low
Patterns: ['Anomaly']
G
